# Adaptive Trust Gate — Full Pipeline (all 7 models)

This notebook contains the **actual implementation code** (not just calls into an installed
package) for the Adaptive Trust Gate project: two experts (CF/SVD++, Content-Based) and five
hybrid gating mechanisms (Static, Learned, Contextual Bandit, GA-Evolved, Sequential BiLSTM)
blended over them, evaluated with segmented RMSE/MAE + ranking metrics + pairwise significance
tests, plus a 5-seed robustness check.

It reproduces the pipeline at
[github.com/Ar555Rathod/adaptive-trust-gate](https://github.com/Ar555Rathod/adaptive-trust-gate)
cell-by-cell, so every model's code is directly visible and editable here rather than imported
as a black box.

Run cells top to bottom. On Colab: **Runtime → Run all**.

In [ ]:
!pip install -q scikit-surprise
# numpy / pandas / scipy / scikit-learn / matplotlib / torch already ship with Colab

## Setup: imports and the MovieLens `ml-latest-small` dataset

Raw data isn't bundled with the code (it's a public, separately-licensed dataset) — this downloads the official small release directly from GroupLens.

In [ ]:
import bisect
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import kendalltau, spearmanr, wilcoxon
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score
from surprise import Dataset, Reader, SVDpp
import matplotlib.pyplot as plt

RAW_DIR = Path("data/raw/ml-latest-small")
PROC_DIR = Path("data/processed")
RESULTS_DIR = Path("results")
MODELS_DIR = RESULTS_DIR / "models"
METRICS_DIR = RESULTS_DIR / "metrics"
PRED_DIR = RESULTS_DIR / "predictions"
for d in (RAW_DIR, PROC_DIR, MODELS_DIR, METRICS_DIR, PRED_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import urllib.request, zipfile

zip_path = Path("data/raw/ml-latest-small.zip")
if not (RAW_DIR / "ratings.csv").exists():
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip", zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("data/raw")
print(sorted(p.name for p in RAW_DIR.iterdir()))

## Config: paths and hyperparameters shared across every model

In [ ]:
RATINGS_CSV = RAW_DIR / "ratings.csv"
MOVIES_CSV = RAW_DIR / "movies.csv"
TAGS_CSV = RAW_DIR / "tags.csv"

RANDOM_SEED = 42
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

# Sparsity segmentation, defined on each user's TRAIN-set rating count
# (cold < 5, warm 5-50, power 50+).
COLD_MAX = 5
WARM_MAX = 50

# Simulated cold-start split: MovieLens guarantees every user has >=20
# ratings, so a plain random 70/15/15 split almost never leaves any user
# with <5 train ratings, leaving the "cold" segment structurally empty. To
# get genuine cold-start cases, a quota of real users have their TRAIN
# allocation deliberately capped to a handful of their own ratings; everyone
# else gets a standard per-user 70/15/15 split. No ratings are fabricated.
COLD_USER_FRAC = 0.15
COLD_TRAIN_MIN = 1
COLD_TRAIN_MAX = 4

# Model 7 (Sequential BiLSTM Gate): a query needs at least this many prior
# TRAIN ratings (strictly before its timestamp) to run the BiLSTM at all;
# below it, it falls back to Model 3's fixed alpha.
SEQ_MIN_HISTORY = 5
SEQ_LONG_LEN = 50
SEQ_SHORT_LEN = 5
RECENCY_HALF_LIFE_DAYS = 180.0

RATING_MIN, RATING_MAX = 0.5, 5.0

## Cold-start-aware train/val/test split, and user sparsity segments

In [ ]:
def build_splits(seed: int = RANDOM_SEED):
    ratings = pd.read_csv(RATINGS_CSV)
    rng = np.random.default_rng(seed)

    users = ratings["userId"].unique()
    shuffled_users = rng.permutation(users)
    n_cold_users = int(round(len(shuffled_users) * COLD_USER_FRAC))
    cold_users = set(shuffled_users[:n_cold_users])

    train_parts, val_parts, test_parts = [], [], []
    for uid, group in ratings.groupby("userId", sort=True):
        order = rng.permutation(len(group))
        group = group.iloc[order]
        n = len(group)

        if uid in cold_users:
            n_train = int(rng.integers(COLD_TRAIN_MIN, COLD_TRAIN_MAX + 1))
            n_train = min(n_train, n)
            remaining = group.iloc[n_train:]
            n_val = len(remaining) // 2
            train_g = group.iloc[:n_train]
            val_g = remaining.iloc[:n_val]
            test_g = remaining.iloc[n_val:]
        else:
            n_train = int(round(n * TRAIN_FRAC))
            n_val = int(round(n * VAL_FRAC))
            train_g = group.iloc[:n_train]
            val_g = group.iloc[n_train: n_train + n_val]
            test_g = group.iloc[n_train + n_val:]

        train_parts.append(train_g)
        val_parts.append(val_g)
        test_parts.append(test_g)

    train_df = pd.concat(train_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    val_df = pd.concat(val_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    test_df = pd.concat(test_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    return train_df, val_df, test_df


def segment_for_count(count: int) -> str:
    if count < COLD_MAX:
        return "cold"
    if count < WARM_MAX:
        return "warm"
    return "power"


def build_user_segments(train_df: pd.DataFrame) -> pd.DataFrame:
    counts = train_df.groupby("userId").size().rename("train_rating_count").reset_index()
    counts["segment"] = counts["train_rating_count"].apply(segment_for_count)
    return counts


def attach_segments(df: pd.DataFrame, user_segments: pd.DataFrame) -> pd.DataFrame:
    out = df.merge(user_segments, on="userId", how="left")
    out["train_rating_count"] = out["train_rating_count"].fillna(0).astype(int)
    out["segment"] = out["segment"].fillna("cold")
    return out

In [ ]:
train_df, val_df, test_df = build_splits(seed=RANDOM_SEED)
train_df.to_csv(PROC_DIR / "train.csv", index=False)
val_df.to_csv(PROC_DIR / "val.csv", index=False)
test_df.to_csv(PROC_DIR / "test.csv", index=False)

user_segments = build_user_segments(train_df)
user_segments.to_csv(PROC_DIR / "user_segments.csv", index=False)

n = len(train_df) + len(val_df) + len(test_df)
print(f"Total ratings: {n}")
print(f"  train: {len(train_df):>6} ({len(train_df)/n:.1%})")
print(f"  val:   {len(val_df):>6} ({len(val_df)/n:.1%})")
print(f"  test:  {len(test_df):>6} ({len(test_df)/n:.1%})")
print("\nUser sparsity segments (by TRAIN rating count):")
print(user_segments["segment"].value_counts())

## Evaluation: RMSE/MAE, ranking metrics, statistical significance

In [ ]:
def rmse(y_true, y_pred) -> float:
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true, y_pred) -> float:
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))


def rating_metrics(y_true, y_pred) -> dict:
    return {"rmse": rmse(y_true, y_pred), "mae": mae(y_true, y_pred), "n": int(len(y_true))}


def segmented_rating_metrics(df, true_col, pred_col, segment_col="segment") -> dict:
    out = {"overall": rating_metrics(df[true_col], df[pred_col])}
    for seg, sub in df.groupby(segment_col):
        out[seg] = rating_metrics(sub[true_col], sub[pred_col])
    return out

In [ ]:
def mean_rank_correlation(df, true_col, pred_col, user_col="userId", min_items=2) -> dict:
    rhos, taus, n_users = [], [], 0
    for _, g in df.groupby(user_col):
        if len(g) < min_items or g[true_col].nunique() < 2 or g[pred_col].nunique() < 2:
            continue
        rho, _ = spearmanr(g[true_col], g[pred_col])
        tau, _ = kendalltau(g[true_col], g[pred_col])
        if np.isnan(rho) or np.isnan(tau):
            continue
        rhos.append(rho); taus.append(tau); n_users += 1
    return {"spearman": float(np.mean(rhos)) if rhos else float("nan"),
            "kendall": float(np.mean(taus)) if taus else float("nan"), "n_users": n_users}


def _dcg(relevances, k, gain="linear"):
    relevances = relevances[:k]
    discounts = np.log2(np.arange(2, len(relevances) + 2))
    gains = relevances if gain == "linear" else (2.0 ** relevances - 1.0)
    return float(np.sum(gains / discounts))


def ndcg_at_k(df, true_col, pred_col, user_col="userId", k=10, gain="linear", min_items=2) -> dict:
    scores, n_users = [], 0
    for _, g in df.groupby(user_col):
        if len(g) < min_items:
            continue
        ranked = g.sort_values(pred_col, ascending=False)[true_col].to_numpy(dtype=float)
        ideal = np.sort(g[true_col].to_numpy(dtype=float))[::-1]
        idcg = _dcg(ideal, k, gain)
        if idcg <= 0:
            continue
        scores.append(_dcg(ranked, k, gain) / idcg); n_users += 1
    return {"ndcg": float(np.mean(scores)) if scores else float("nan"), "k": k, "n_users": n_users}


def arhr_at_k(df, true_col, pred_col, user_col="userId", k=10, relevance_threshold=4.0) -> dict:
    scores, n_users = [], 0
    for _, g in df.groupby(user_col):
        relevant_mask = g[true_col].to_numpy(dtype=float) >= relevance_threshold
        if not relevant_mask.any():
            continue
        order = g[pred_col].to_numpy(dtype=float).argsort()[::-1][:k]
        ranked_relevant = relevant_mask[order]
        hit_positions = np.flatnonzero(ranked_relevant)
        rr = 1.0 / (hit_positions[0] + 1) if len(hit_positions) > 0 else 0.0
        scores.append(rr); n_users += 1
    return {"arhr": float(np.mean(scores)) if scores else float("nan"), "k": k,
            "relevance_threshold": relevance_threshold, "n_users": n_users}


def roc_auc(df, true_col, pred_col, relevance_threshold=4.0) -> dict:
    y = (df[true_col].to_numpy(dtype=float) >= relevance_threshold).astype(int)
    if y.sum() == 0 or y.sum() == len(y):
        return {"roc_auc": float("nan"), "relevance_threshold": relevance_threshold, "n": int(len(y))}
    auc = roc_auc_score(y, df[pred_col].to_numpy(dtype=float))
    return {"roc_auc": float(auc), "relevance_threshold": relevance_threshold, "n": int(len(y))}


def ranking_metrics(df, true_col, pred_col, user_col="userId", k=10, relevance_threshold=4.0) -> dict:
    out = {}
    out.update(mean_rank_correlation(df, true_col, pred_col, user_col))
    out.update(ndcg_at_k(df, true_col, pred_col, user_col, k=k))
    out.update(arhr_at_k(df, true_col, pred_col, user_col, k=k, relevance_threshold=relevance_threshold))
    out.update(roc_auc(df, true_col, pred_col, relevance_threshold=relevance_threshold))
    return out


def segmented_ranking_metrics(df, true_col, pred_col, user_col="userId", segment_col="segment",
                               k=10, relevance_threshold=4.0) -> dict:
    out = {"overall": ranking_metrics(df, true_col, pred_col, user_col, k, relevance_threshold)}
    for seg, sub in df.groupby(segment_col):
        out[seg] = ranking_metrics(sub, true_col, pred_col, user_col, k, relevance_threshold)
    return out

In [ ]:
def paired_bootstrap_rmse_diff(y_true, pred_a, pred_b, n_boot=2000, seed=42, ci=0.95) -> dict:
    y_true, pred_a, pred_b = (np.asarray(a, dtype=float) for a in (y_true, pred_a, pred_b))
    n = len(y_true)
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        diffs[i] = rmse(y_true[idx], pred_a[idx]) - rmse(y_true[idx], pred_b[idx])
    alpha = (1 - ci) / 2
    lo, hi = np.percentile(diffs, [alpha * 100, (1 - alpha) * 100])
    p_value = 2 * min(np.mean(diffs >= 0), np.mean(diffs <= 0))
    return {"rmse_a": rmse(y_true, pred_a), "rmse_b": rmse(y_true, pred_b),
            "mean_diff_a_minus_b": float(diffs.mean()), "ci_lo": float(lo), "ci_hi": float(hi),
            "ci_level": ci, "significant": bool(lo > 0 or hi < 0),
            "p_value_approx": float(min(p_value, 1.0)), "n_boot": n_boot, "n": n}


def wilcoxon_paired(y_true, pred_a, pred_b) -> dict:
    y_true = np.asarray(y_true, dtype=float)
    err_a = (y_true - np.asarray(pred_a, dtype=float)) ** 2
    err_b = (y_true - np.asarray(pred_b, dtype=float)) ** 2
    diff = err_a - err_b
    if np.allclose(diff, 0):
        return {"statistic": 0.0, "p_value": 1.0, "n": int(len(y_true))}
    stat, p = wilcoxon(err_a, err_b)
    return {"statistic": float(stat), "p_value": float(p), "n": int(len(y_true))}


def compare_models(df, true_col, model_preds, segment_col=None, n_boot=2000, seed=42) -> pd.DataFrame:
    rows = []
    groups = [("overall", df)] if segment_col is None else [("overall", df)] + list(df.groupby(segment_col))
    names = list(model_preds.keys())
    for seg_name, sub in groups:
        y = sub[true_col].to_numpy(dtype=float)
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                a, b = names[i], names[j]
                pa = sub[model_preds[a]].to_numpy(dtype=float)
                pb = sub[model_preds[b]].to_numpy(dtype=float)
                boot = paired_bootstrap_rmse_diff(y, pa, pb, n_boot=n_boot, seed=seed)
                wil = wilcoxon_paired(y, pa, pb)
                rows.append({"segment": seg_name, "model_a": a, "model_b": b,
                             "rmse_a": boot["rmse_a"], "rmse_b": boot["rmse_b"],
                             "mean_diff": boot["mean_diff_a_minus_b"], "ci_lo": boot["ci_lo"],
                             "ci_hi": boot["ci_hi"], "bootstrap_significant": boot["significant"],
                             "wilcoxon_p": wil["p_value"], "wilcoxon_significant": wil["p_value"] < 0.05,
                             "n": boot["n"]})
    return pd.DataFrame(rows)

## Model 1 — Collaborative Filtering expert (SVD++)

In [ ]:
class CFExpertSVDpp:
    def __init__(self, n_factors=20, n_epochs=20, random_state=RANDOM_SEED):
        self.n_factors, self.n_epochs, self.random_state = n_factors, n_epochs, random_state
        self.algo = SVDpp(n_factors=n_factors, n_epochs=n_epochs, random_state=random_state)
        self.train_time_sec = None
        self.global_mean_ = None

    def fit(self, train_df: pd.DataFrame) -> "CFExpertSVDpp":
        reader = Reader(rating_scale=(RATING_MIN, RATING_MAX))
        data = Dataset.load_from_df(train_df[["userId", "movieId", "rating"]], reader)
        trainset = data.build_full_trainset()
        start = time.perf_counter()
        self.algo.fit(trainset)
        self.train_time_sec = time.perf_counter() - start
        self.global_mean_ = trainset.global_mean
        return self

    def predict(self, user_id, item_id) -> float:
        return float(self.algo.predict(user_id, item_id, clip=True).est)

    def predict_batch(self, df, user_col="userId", item_col="movieId") -> np.ndarray:
        return np.array([self.predict(u, i) for u, i in zip(df[user_col], df[item_col])])

    def is_known_user(self, user_id) -> bool:
        try:
            self.algo.trainset.to_inner_uid(user_id); return True
        except ValueError:
            return False

    def is_known_item(self, item_id) -> bool:
        try:
            self.algo.trainset.to_inner_iid(item_id); return True
        except ValueError:
            return False

    def n_params(self) -> int:
        trainset = self.algo.trainset
        n_users, n_items, f = trainset.n_users, trainset.n_items, self.n_factors
        return n_users + n_items + n_users * f + n_items * f + n_items * f

## Model 2 — Content-Based expert (TF-IDF genres/tags + item-item cosine kNN)

In [ ]:
def _soup(genres: str, tags: str) -> str:
    genre_tokens = genres.replace("|", " ").replace("(no genres listed)", "")
    return f"{genre_tokens} {tags}".strip()


class ContentBasedExpert:
    def __init__(self, top_k: int = 20):
        self.top_k = top_k
        self.vectorizer = TfidfVectorizer(token_pattern=r"[^\s]+")
        self.train_time_sec = None
        self.global_mean_ = None
        self.user_mean_ = {}
        self.item_index_ = {}
        self.item_vectors = None
        self.user_train_history_ = {}

    def fit(self, train_df, movies_df, tags_df=None) -> "ContentBasedExpert":
        start = time.perf_counter()
        tags_per_movie = {}
        if tags_df is not None:
            grouped = tags_df.groupby("movieId")["tag"].apply(lambda s: " ".join(s.astype(str).str.lower()))
            tags_per_movie = grouped.to_dict()

        movies = movies_df.reset_index(drop=True)
        self.item_index_ = {mid: idx for idx, mid in enumerate(movies["movieId"])}
        corpus = [_soup(row.genres, tags_per_movie.get(row.movieId, "")) for row in movies.itertuples(index=False)]
        self.item_vectors = self.vectorizer.fit_transform(corpus)

        self.global_mean_ = float(train_df["rating"].mean())
        self.user_mean_ = train_df.groupby("userId")["rating"].mean().to_dict()
        self.user_train_history_ = {uid: list(zip(g["movieId"], g["rating"])) for uid, g in train_df.groupby("userId")}
        self.train_time_sec = time.perf_counter() - start
        return self

    def _fallback(self, user_id) -> float:
        return self.user_mean_.get(user_id, self.global_mean_)

    def _selected_neighbors(self, user_id, item_id):
        empty = (np.array([]), np.array([]))
        if item_id not in self.item_index_:
            return empty
        history = self.user_train_history_.get(user_id)
        if not history:
            return empty
        target_vec = self.item_vectors[self.item_index_[item_id]]
        hist_item_ids = [mid for mid, _ in history]
        hist_ratings = np.array([r for _, r in history], dtype=float)
        keep = [i for i, mid in enumerate(hist_item_ids) if mid in self.item_index_]
        if not keep:
            return empty
        hist_ratings = hist_ratings[keep]
        hist_rows = [self.item_index_[hist_item_ids[i]] for i in keep]
        hist_vecs = self.item_vectors[hist_rows]
        sims = np.asarray(hist_vecs.dot(target_vec.T).todense()).ravel()
        if self.top_k is not None and len(sims) > self.top_k:
            top_idx = np.argpartition(sims, -self.top_k)[-self.top_k:]
        else:
            top_idx = np.arange(len(sims))
        sel_sims, sel_ratings = sims[top_idx], hist_ratings[top_idx]
        mask = sel_sims > 0
        return sel_sims[mask], sel_ratings[mask]

    def predict(self, user_id, item_id) -> float:
        w, r = self._selected_neighbors(user_id, item_id)
        if len(w) == 0:
            return self._fallback(user_id)
        score = float(np.dot(w, r) / w.sum())
        return float(np.clip(score, RATING_MIN, RATING_MAX))

    def predict_batch(self, df, user_col="userId", item_col="movieId") -> np.ndarray:
        return np.array([self.predict(u, i) for u, i in zip(df[user_col], df[item_col])])

    def similarity_diagnostics(self, user_id, item_id):
        w, _ = self._selected_neighbors(user_id, item_id)
        if len(w) == 0:
            return 0.0, 0, 0.0
        return float(w.max()), int(len(w)), float(w.sum())

    def n_params(self) -> int:
        return int(self.item_vectors.shape[0] * self.item_vectors.shape[1])

### Fit both experts on TRAIN and score VAL/TEST

In [ ]:
movies_df = pd.read_csv(MOVIES_CSV)
tags_df = pd.read_csv(TAGS_CSV)

print("Training CF expert (SVD++)...")
cf_expert = CFExpertSVDpp().fit(train_df)
print(f"  train_time_sec={cf_expert.train_time_sec:.2f}  n_params={cf_expert.n_params()}")

print("Training Content-Based expert...")
cb_expert = ContentBasedExpert().fit(train_df, movies_df, tags_df)
print(f"  train_time_sec={cb_expert.train_time_sec:.2f}  n_params(repr size)={cb_expert.n_params()}")

experts_results = {}
val_df = attach_segments(val_df.copy(), user_segments)
test_df = attach_segments(test_df.copy(), user_segments)
for split_name, df in (("val", val_df), ("test", test_df)):
    df["cf_pred"] = cf_expert.predict_batch(df)
    df["cb_pred"] = cb_expert.predict_batch(df)

    cf_metrics = segmented_rating_metrics(df, "rating", "cf_pred")
    cb_metrics = segmented_rating_metrics(df, "rating", "cb_pred")
    experts_results[split_name] = {"cf_svdpp": cf_metrics, "content_based": cb_metrics}
    print(f"\n[{split_name}] CF RMSE(overall)={cf_metrics['overall']['rmse']:.4f}  "
          f"CB RMSE(overall)={cb_metrics['overall']['rmse']:.4f}")

    df.to_csv(PRED_DIR / f"experts_{split_name}.csv", index=False)

experts_results["compute_cost"] = {
    "cf_svdpp": {"train_time_sec": cf_expert.train_time_sec, "n_params": cf_expert.n_params()},
    "content_based": {"train_time_sec": cb_expert.train_time_sec, "n_params_repr_size": cb_expert.n_params()},
}
with open(METRICS_DIR / "experts_baseline.json", "w") as f:
    json.dump(experts_results, f, indent=2)

## Shared hybrid blend formula

`r_hat(u,i) = g(u,i) * CF_score(u,i) + (1 - g(u,i)) * CB_score(u,i)` — only `g` changes between Models 3-7; the combiner itself stays fixed so the comparison is apples-to-apples.

In [ ]:
def blend_scores(cf_scores, cb_scores, g, clip: bool = True):
    cf_scores, cb_scores, g = (np.asarray(a, dtype=float) for a in (cf_scores, cb_scores, g))
    pred = g * cf_scores + (1 - g) * cb_scores
    if clip:
        pred = np.clip(pred, RATING_MIN, RATING_MAX)
    return pred

## Model 3 — Static Hybrid gate: g(u,i) = alpha (grid-searched constant)

In [ ]:
class StaticGate:
    def __init__(self, alpha=None):
        self.alpha = alpha
        self.train_time_sec = None
        self.grid_curve_ = None

    def g(self, n: int) -> np.ndarray:
        return np.full(n, self.alpha, dtype=float)

    def fit(self, val_df, cf_col="cf_pred", cb_col="cb_pred", true_col="rating", step=0.01, metric="rmse"):
        start = time.perf_counter()
        alphas = np.arange(0.0, 1.0 + step / 2, step)
        cf = val_df[cf_col].to_numpy(dtype=float)
        cb = val_df[cb_col].to_numpy(dtype=float)
        y = val_df[true_col].to_numpy(dtype=float)
        curve, best_alpha, best_score = [], None, np.inf
        for a in alphas:
            pred = np.clip(a * cf + (1 - a) * cb, 0.5, 5.0)
            r, m = rmse(y, pred), mae(y, pred)
            curve.append((float(a), r, m))
            score = r if metric == "rmse" else m
            if score < best_score:
                best_score, best_alpha = score, float(a)
        self.alpha, self.grid_curve_ = best_alpha, curve
        self.train_time_sec = time.perf_counter() - start
        return self

    def n_params(self) -> int:
        return 1

In [ ]:
static_gate = StaticGate().fit(val_df, step=0.01, metric="rmse")
print(f"Best alpha={static_gate.alpha:.2f}  (train_time_sec={static_gate.train_time_sec:.3f})")
pd.DataFrame(static_gate.grid_curve_, columns=["alpha", "rmse", "mae"]).to_csv(
    METRICS_DIR / "model3_alpha_grid.csv", index=False)

val_df["hybrid_pred"] = blend_scores(val_df["cf_pred"], val_df["cb_pred"], static_gate.g(len(val_df)))
test_df["hybrid_pred"] = blend_scores(test_df["cf_pred"], test_df["cb_pred"], static_gate.g(len(test_df)))
model3_val_metrics = segmented_rating_metrics(val_df, "rating", "hybrid_pred")
model3_test_metrics = segmented_rating_metrics(test_df, "rating", "hybrid_pred")
print(f"[test] RMSE(overall)={model3_test_metrics['overall']['rmse']:.4f}")

test_df.to_csv(PRED_DIR / "model3_static_hybrid_test.csv", index=False)
with open(METRICS_DIR / "model3_static_hybrid.json", "w") as f:
    json.dump({"alpha": static_gate.alpha, "val": model3_val_metrics, "test": model3_test_metrics,
               "compute_cost": {"train_time_sec": static_gate.train_time_sec, "n_params": static_gate.n_params()}},
              f, indent=2)

## Gate context features (shared by Models 4, 5, 6, 7)

Sparsity + content-similarity + CF-confidence signals the adaptive gates condition `g(u,i)` on. Every feature is derived from either TRAIN-only counts/history or the frozen experts' own predictions on the row being scored — no val/test label ever leaks into a feature.

In [ ]:
FEATURE_COLUMNS = [
    "log_user_count", "log_item_count", "cf_known_user", "cf_known_item",
    "cb_max_sim", "cb_support", "cb_sim_weight_sum", "pred_gap", "abs_pred_gap",
]


def build_item_popularity(train_df: pd.DataFrame) -> dict:
    return train_df.groupby("movieId").size().to_dict()


def build_gate_features(df, item_popularity, cf_expert, cb_expert, user_col="userId", item_col="movieId") -> pd.DataFrame:
    user_counts = df["train_rating_count"].to_numpy(dtype=float)
    item_counts = np.array([item_popularity.get(i, 0) for i in df[item_col]], dtype=float)
    cf_known_user = np.array([cf_expert.is_known_user(u) for u in df[user_col]], dtype=float)
    cf_known_item = np.array([cf_expert.is_known_item(i) for i in df[item_col]], dtype=float)
    diagnostics = [cb_expert.similarity_diagnostics(u, i) for u, i in zip(df[user_col], df[item_col])]
    cb_max_sim = np.array([d[0] for d in diagnostics], dtype=float)
    cb_support = np.array([d[1] for d in diagnostics], dtype=float)
    cb_sim_weight_sum = np.array([d[2] for d in diagnostics], dtype=float)
    cf_pred = df["cf_pred"].to_numpy(dtype=float)
    cb_pred = df["cb_pred"].to_numpy(dtype=float)
    feats = pd.DataFrame({
        "log_user_count": np.log1p(user_counts), "log_item_count": np.log1p(item_counts),
        "cf_known_user": cf_known_user, "cf_known_item": cf_known_item,
        "cb_max_sim": cb_max_sim, "cb_support": cb_support, "cb_sim_weight_sum": cb_sim_weight_sum,
        "pred_gap": cf_pred - cb_pred, "abs_pred_gap": np.abs(cf_pred - cb_pred),
    })
    return feats[FEATURE_COLUMNS]


item_popularity = build_item_popularity(train_df)

## GateNet — the small network Models 4 and 6 both use for `g = f(x)`

`hidden_size=0` is a linear/logistic-regression-capacity gate solved in one shot via ridge weighted least squares (exact, no local optima). `hidden_size>0` is a genuine one-hidden-layer MLP (tanh → sigmoid) trained with Adam, used when Model 6's GA search picks nonzero capacity. Both are trained end-to-end against the *hybrid* blend loss `L = mean((g*cf + (1-g)*cb - y)^2) + l2*||weights||^2`, not a proxy target for g itself.

In [ ]:
def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))


class GateNet:
    def __init__(self, n_features, hidden_size=0, l2=1.0, lr=0.05, seed=42):
        self.n_features, self.hidden_size, self.l2, self.lr = n_features, hidden_size, l2, lr
        rng = np.random.default_rng(seed)
        scale1 = np.sqrt(2.0 / max(n_features, 1))
        if hidden_size > 0:
            self.W1 = rng.normal(0, scale1, size=(n_features, hidden_size))
            self.b1 = np.zeros(hidden_size)
            scale2 = np.sqrt(2.0 / max(hidden_size, 1))
            self.W2 = rng.normal(0, scale2, size=(hidden_size, 1))
        else:
            self.W1 = None; self.b1 = None
            self.W2 = np.zeros((n_features, 1))
        self.b2 = np.zeros(1)
        self.feat_mean_ = np.zeros(n_features)
        self.feat_std_ = np.ones(n_features)
        self._adam_state = None
        self.history_ = []

    def _normalize(self, X):
        return (X - self.feat_mean_) / self.feat_std_

    def _forward(self, Xn):
        if self.hidden_size > 0:
            Z1 = Xn @ self.W1 + self.b1
            A1 = np.tanh(Z1)
            Z2 = A1 @ self.W2 + self.b2
            g = _sigmoid(Z2).ravel()
        else:
            A1 = None
            Z2 = Xn @ self.W2 + self.b2
            g = np.clip(Z2, 0.0, 1.0).ravel()
        return g, A1

    def g(self, X) -> np.ndarray:
        Xn = self._normalize(np.asarray(X, dtype=float))
        g, _ = self._forward(Xn)
        return g

    def n_params(self) -> int:
        n = self.W2.size + self.b2.size
        if self.hidden_size > 0:
            n += self.W1.size + self.b1.size
        return int(n)

    def _hybrid_loss(self, X, cf, cb, y):
        g = self.g(X)
        pred = g * cf + (1 - g) * cb
        return float(np.mean((pred - y) ** 2))

    def _fit_linear_closed_form(self, X, cf, cb, y, X_es, cf_es, cb_es, y_es):
        Xn = self._normalize(X)
        Xd = np.hstack([Xn, np.ones((len(Xn), 1))])
        diff = cf - cb
        r = y - cb
        Xu = Xd * diff[:, None]
        A = Xu.T @ Xu + self.l2 * np.eye(Xd.shape[1])
        b = Xu.T @ r
        beta = np.linalg.solve(A, b)
        self.W2 = beta[:-1].reshape(-1, 1)
        self.b2 = beta[-1:]
        train_loss = self._hybrid_loss(X, cf, cb, y)
        es_loss = self._hybrid_loss(X_es, cf_es, cb_es, y_es)
        self.history_ = [(1, train_loss, es_loss)]

    def _params_and_grads(self, Xn, A1, dL_dz2):
        n = Xn.shape[0]
        grads = {"W2": A1.T @ dL_dz2 / n + self.l2 * self.W2, "b2": dL_dz2.mean(axis=0)}
        dL_dA1 = dL_dz2 @ self.W2.T
        dL_dz1 = dL_dA1 * (1 - A1 ** 2)
        grads["W1"] = Xn.T @ dL_dz1 / n + self.l2 * self.W1
        grads["b1"] = dL_dz1.mean(axis=0)
        return grads

    def _init_adam(self):
        self._adam_state = {p: {"m": np.zeros_like(getattr(self, p)), "v": np.zeros_like(getattr(self, p)), "t": 0}
                             for p in ["W1", "b1", "W2", "b2"]}

    def _adam_step(self, grads, beta1=0.9, beta2=0.999, eps=1e-8):
        for name, grad in grads.items():
            st = self._adam_state[name]
            st["t"] += 1
            st["m"] = beta1 * st["m"] + (1 - beta1) * grad
            st["v"] = beta2 * st["v"] + (1 - beta2) * (grad ** 2)
            m_hat = st["m"] / (1 - beta1 ** st["t"])
            v_hat = st["v"] / (1 - beta2 ** st["t"])
            update = self.lr * m_hat / (np.sqrt(v_hat) + eps)
            setattr(self, name, getattr(self, name) - update)

    def _fit_mlp_gd(self, X, cf, cb, y, X_es, cf_es, cb_es, y_es, epochs, patience, log_every):
        Xn = self._normalize(X)
        self._init_adam()
        best_es_loss, best_state, no_improve = np.inf, None, 0
        for epoch in range(1, epochs + 1):
            g, A1 = self._forward(Xn)
            pred = g * cf + (1 - g) * cb
            n = len(y)
            dL_dpred = 2.0 * (pred - y) / n
            dpred_dg = cf - cb
            dL_dg = dL_dpred * dpred_dg
            dg_dz2 = g * (1 - g)
            dL_dz2 = (dL_dg * dg_dz2).reshape(-1, 1)
            grads = self._params_and_grads(Xn, A1, dL_dz2)
            self._adam_step(grads)
            if epoch % log_every == 0 or epoch == epochs:
                train_loss = self._hybrid_loss(X, cf, cb, y)
                es_loss = self._hybrid_loss(X_es, cf_es, cb_es, y_es)
                self.history_.append((epoch, train_loss, es_loss))
                if es_loss < best_es_loss - 1e-6:
                    best_es_loss, best_state, no_improve = es_loss, self._snapshot(), 0
                else:
                    no_improve += 1
                    if no_improve * log_every >= patience:
                        break
        if best_state is not None:
            self._restore(best_state)

    def fit(self, X, cf, cb, y, X_es, cf_es, cb_es, y_es, epochs=3000, patience=200, log_every=100):
        X, cf, cb, y = (np.asarray(a, dtype=float) for a in (X, cf, cb, y))
        X_es, cf_es, cb_es, y_es = (np.asarray(a, dtype=float) for a in (X_es, cf_es, cb_es, y_es))
        self.feat_mean_ = X.mean(axis=0)
        self.feat_std_ = X.std(axis=0)
        self.feat_std_[self.feat_std_ < 1e-8] = 1.0
        if self.hidden_size == 0:
            self._fit_linear_closed_form(X, cf, cb, y, X_es, cf_es, cb_es, y_es)
        else:
            self._fit_mlp_gd(X, cf, cb, y, X_es, cf_es, cb_es, y_es, epochs, patience, log_every)
        return self

    def _snapshot(self):
        return {"W1": self.W1.copy(), "b1": self.b1.copy(), "W2": self.W2.copy(), "b2": self.b2.copy()}

    def _restore(self, state):
        for k, v in state.items():
            setattr(self, k, v)

## Model 4 — Learned Gate

A `GateNet(hidden_size=0)` fit on VAL (the experts' honest out-of-sample predictions), evaluated on TEST — never fit on TRAIN, which SVD++ has partly memorized.

In [ ]:
class LearnedGate:
    def __init__(self, hidden_size=0, l2=1.0, lr=0.05, es_frac=0.2, seed=42):
        self.hidden_size, self.l2, self.lr, self.es_frac, self.seed = hidden_size, l2, lr, es_frac, seed
        self.net = None
        self.train_time_sec = None

    def fit(self, val_df, item_popularity, cf_expert, cb_expert, epochs=3000, patience=200):
        start = time.perf_counter()
        feats = build_gate_features(val_df, item_popularity, cf_expert, cb_expert)
        X = feats.to_numpy(dtype=float)
        cf = val_df["cf_pred"].to_numpy(dtype=float)
        cb = val_df["cb_pred"].to_numpy(dtype=float)
        y = val_df["rating"].to_numpy(dtype=float)
        rng = np.random.default_rng(self.seed)
        idx = rng.permutation(len(val_df))
        n_es = int(round(len(val_df) * self.es_frac))
        es_idx, tr_idx = idx[:n_es], idx[n_es:]
        self.net = GateNet(n_features=X.shape[1], hidden_size=self.hidden_size, l2=self.l2, lr=self.lr, seed=self.seed)
        self.net.fit(X[tr_idx], cf[tr_idx], cb[tr_idx], y[tr_idx],
                     X[es_idx], cf[es_idx], cb[es_idx], y[es_idx], epochs=epochs, patience=patience)
        self.train_time_sec = time.perf_counter() - start
        return self

    def g(self, df, item_popularity, cf_expert, cb_expert) -> np.ndarray:
        feats = build_gate_features(df, item_popularity, cf_expert, cb_expert)
        return self.net.g(feats.to_numpy(dtype=float))

    def n_params(self) -> int:
        return self.net.n_params()

In [ ]:
print("Fitting Learned Gate (logistic-regression capacity) on VAL...")
learned_gate = LearnedGate(hidden_size=0, l2=1e-3, lr=0.05, es_frac=0.2).fit(
    val_df, item_popularity, cf_expert, cb_expert, epochs=3000, patience=200)
print(f"  train_time_sec={learned_gate.train_time_sec:.2f}  n_params={learned_gate.n_params()}")

val_df["gate_g"] = learned_gate.g(val_df, item_popularity, cf_expert, cb_expert)
val_df["hybrid_pred"] = blend_scores(val_df["cf_pred"], val_df["cb_pred"], val_df["gate_g"])
test_df["gate_g"] = learned_gate.g(test_df, item_popularity, cf_expert, cb_expert)
test_df["hybrid_pred"] = blend_scores(test_df["cf_pred"], test_df["cb_pred"], test_df["gate_g"])

model4_val_metrics = segmented_rating_metrics(val_df, "rating", "hybrid_pred")
model4_test_metrics = segmented_rating_metrics(test_df, "rating", "hybrid_pred")
print(f"[test] RMSE(overall)={model4_test_metrics['overall']['rmse']:.4f}  "
      f"mean g by segment: {test_df.groupby('segment')['gate_g'].mean().to_dict()}")

test_df.to_csv(PRED_DIR / "model4_learned_gate_test.csv", index=False)
with open(METRICS_DIR / "model4_learned_gate.json", "w") as f:
    json.dump({"val": model4_val_metrics, "test": model4_test_metrics,
               "compute_cost": {"train_time_sec": learned_gate.train_time_sec, "n_params": learned_gate.n_params()},
               "training_curve_tail": learned_gate.net.history_[-5:]}, f, indent=2)

## Model 5 — Contextual Bandit Gate (LinUCB)

Selects a blend weight from a discrete set of arms per (u,i) using the same context features, updating a disjoint ridge reward model online after every interaction — the actual escalation over Model 4 is the continuous online update loop, not a bigger context.

In [ ]:
class ContextualBanditGate:
    def __init__(self, n_features, arms=None, strategy="ucb", ucb_alpha=1.0, epsilon=0.1,
                 ridge_lambda=1.0, seed=42):
        self.arms = np.asarray(arms if arms is not None else np.linspace(0.0, 1.0, 11))
        self.n_arms = len(self.arms)
        self.d = n_features + 1
        self.strategy, self.ucb_alpha, self.epsilon, self.ridge_lambda = strategy, ucb_alpha, epsilon, ridge_lambda
        self.rng = np.random.default_rng(seed)
        self.A_inv = np.stack([np.eye(self.d) / ridge_lambda for _ in range(self.n_arms)])
        self.b = np.zeros((self.n_arms, self.d))
        self.n_pulls = np.zeros(self.n_arms, dtype=int)

    def _augment(self, x):
        return np.concatenate([np.asarray(x, dtype=float), [1.0]])

    def _theta(self, k):
        return self.A_inv[k] @ self.b[k]

    def select_arm(self, x, explore=True) -> int:
        xa = self._augment(x)
        if self.strategy == "epsilon_greedy":
            if explore and self.rng.random() < self.epsilon:
                return int(self.rng.integers(self.n_arms))
            scores = np.array([xa @ self._theta(k) for k in range(self.n_arms)])
            return int(np.argmax(scores))
        scores = np.empty(self.n_arms)
        for k in range(self.n_arms):
            mean_k = xa @ self._theta(k)
            bonus_k = self.ucb_alpha * np.sqrt(max(float(xa @ self.A_inv[k] @ xa), 0.0)) if explore else 0.0
            scores[k] = mean_k + bonus_k
        return int(np.argmax(scores))

    def update(self, k, x, reward) -> None:
        xa = self._augment(x)
        Ainv = self.A_inv[k]
        Ainv_x = Ainv @ xa
        denom = 1.0 + xa @ Ainv_x
        self.A_inv[k] = Ainv - np.outer(Ainv_x, Ainv_x) / denom
        self.b[k] += reward * xa
        self.n_pulls[k] += 1

    def n_params(self) -> int:
        return int(self.A_inv.size + self.b.size)


def run_stream(bandit, X, cf, cb, y, explore=True, clip=(0.5, 5.0)):
    n = len(y)
    preds, chosen_alpha, sq_err = np.empty(n), np.empty(n), np.empty(n)
    for t in range(n):
        x = X[t]
        k = bandit.select_arm(x, explore=explore)
        alpha = bandit.arms[k]
        pred = float(np.clip(alpha * cf[t] + (1 - alpha) * cb[t], *clip))
        reward = -((pred - y[t]) ** 2)
        if explore:
            bandit.update(k, x, reward)
        preds[t], chosen_alpha[t], sq_err[t] = pred, alpha, (pred - y[t]) ** 2
    return preds, chosen_alpha, sq_err

In [ ]:
# Phase 1: "deployed" eval -- learn online over VAL (timestamp order), freeze, score on TEST.
val_sorted = val_df.sort_values("timestamp").reset_index(drop=True)
X_val = build_gate_features(val_sorted, item_popularity, cf_expert, cb_expert).to_numpy(dtype=float)

print("Training Bandit Gate (LinUCB) online over VAL (timestamp order)...")
start = time.perf_counter()
bandit = ContextualBanditGate(n_features=len(FEATURE_COLUMNS), strategy="ucb", ucb_alpha=1.0, seed=RANDOM_SEED)
run_stream(bandit, X_val, val_sorted["cf_pred"].to_numpy(dtype=float), val_sorted["cb_pred"].to_numpy(dtype=float),
           val_sorted["rating"].to_numpy(dtype=float), explore=True)
bandit_train_time_sec = time.perf_counter() - start
print(f"  train_time_sec={bandit_train_time_sec:.2f}  arm_pulls={bandit.n_pulls.tolist()}")

X_test = build_gate_features(test_df, item_popularity, cf_expert, cb_expert).to_numpy(dtype=float)
test_preds, test_alphas, _ = run_stream(
    bandit, X_test, test_df["cf_pred"].to_numpy(dtype=float), test_df["cb_pred"].to_numpy(dtype=float),
    test_df["rating"].to_numpy(dtype=float), explore=False)
test_df["hybrid_pred"], test_df["gate_g"] = test_preds, test_alphas
model5_test_metrics = segmented_rating_metrics(test_df, "rating", "hybrid_pred")
print(f"[test, frozen] RMSE(overall)={model5_test_metrics['overall']['rmse']:.4f}")

test_df.to_csv(PRED_DIR / "model5_bandit_gate_test.csv", index=False)
with open(METRICS_DIR / "model5_bandit_gate.json", "w") as f:
    json.dump({"test": model5_test_metrics,
               "compute_cost": {"train_time_sec": bandit_train_time_sec, "n_params": bandit.n_params()},
               "arm_alphas": bandit.arms.tolist(), "arm_pulls_on_val": bandit.n_pulls.tolist()}, f, indent=2)

### Model 5 online learning-curve simulation (VAL+TEST streamed chronologically)

In [ ]:
def rolling_rmse(sq_err: np.ndarray, window: int = 500) -> np.ndarray:
    cum = np.cumsum(np.insert(sq_err, 0, 0.0))
    out = np.full(len(sq_err), np.nan)
    for i in range(len(sq_err)):
        lo = max(0, i + 1 - window)
        out[i] = np.sqrt((cum[i + 1] - cum[lo]) / (i + 1 - lo))
    return out


stream_df = pd.concat([val_df, test_df.drop(columns=["hybrid_pred", "gate_g"])], ignore_index=True)
stream_df = stream_df.sort_values("timestamp").reset_index(drop=True)
X_stream = build_gate_features(stream_df, item_popularity, cf_expert, cb_expert).to_numpy(dtype=float)
cf_stream = stream_df["cf_pred"].to_numpy(dtype=float)
cb_stream = stream_df["cb_pred"].to_numpy(dtype=float)
y_stream = stream_df["rating"].to_numpy(dtype=float)

curves = {}
for strategy, label in (("ucb", "Bandit (LinUCB)"), ("epsilon_greedy", "Bandit (epsilon-greedy)")):
    b = ContextualBanditGate(n_features=len(FEATURE_COLUMNS), strategy=strategy, ucb_alpha=1.0, epsilon=0.1, seed=RANDOM_SEED)
    _, _, sq_err = run_stream(b, X_stream, cf_stream, cb_stream, y_stream, explore=True)
    curves[label] = sq_err

curves["CF/SVD++ only"] = (cf_stream - y_stream) ** 2
curves["Content-Based only"] = (cb_stream - y_stream) ** 2
static_pred = np.clip(static_gate.alpha * cf_stream + (1 - static_gate.alpha) * cb_stream, 0.5, 5.0)
curves[f"Static Hybrid (a={static_gate.alpha:.2f})"] = (static_pred - y_stream) ** 2

window = 500
curve_df = pd.DataFrame({"step": np.arange(1, len(y_stream) + 1)})
for label, sq_err in curves.items():
    curve_df[label] = rolling_rmse(sq_err, window=window)
curve_df.to_csv(METRICS_DIR / "model5_learning_curve.csv", index=False)

plt.figure(figsize=(9, 5.5))
for label in curves:
    plt.plot(curve_df["step"], curve_df[label], label=label, linewidth=1.3)
plt.xlabel(f"Interaction step (chronological, VAL+TEST, n={len(y_stream)})")
plt.ylabel(f"Rolling RMSE (window={window})")
plt.title("Model 5: Bandit Gate online learning curve")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(METRICS_DIR / "model5_learning_curve.png", dpi=150)
plt.show()

## Model 6 — GA-Evolved Gate

A genetic algorithm searches over which of the 9 gate features to use and how much capacity the gate needs (`hidden_size`), rather than hand-picking Model 4's full feature set. Each candidate genome is trained with the same `GateNet` class Model 4 uses, on the same VAL split, for a fair fitness comparison; the winning genome is retrained to full convergence for the final gate.

In [ ]:
HIDDEN_SIZE_CHOICES = (0, 4, 8)


def _random_genome(rng, n_features):
    mask = rng.random(n_features) < 0.6
    if not mask.any():
        mask[rng.integers(n_features)] = True
    return {"mask": mask, "hidden_size": int(rng.choice(HIDDEN_SIZE_CHOICES))}


def _fit_genome(genome, X_tr, cf_tr, cb_tr, y_tr, X_es, cf_es, cb_es, y_es, epochs, patience, seed):
    cols = np.where(genome["mask"])[0]
    l2 = 1.0 if genome["hidden_size"] == 0 else 1e-3
    net = GateNet(n_features=len(cols), hidden_size=genome["hidden_size"], l2=l2, lr=0.05, seed=seed)
    net.fit(X_tr[:, cols], cf_tr, cb_tr, y_tr, X_es[:, cols], cf_es, cb_es, y_es,
            epochs=epochs, patience=patience, log_every=max(epochs // 10, 1))
    return net.history_[-1][2], net


def _crossover(rng, p1, p2):
    mask = np.where(rng.random(len(p1["mask"])) < 0.5, p1["mask"], p2["mask"])
    if not mask.any():
        mask[rng.integers(len(mask))] = True
    hidden = p1["hidden_size"] if rng.random() < 0.5 else p2["hidden_size"]
    return {"mask": mask, "hidden_size": hidden}


def _mutate(rng, genome, feature_flip_prob=0.15, hidden_jump_prob=0.2):
    mask = genome["mask"].copy()
    flips = rng.random(len(mask)) < feature_flip_prob
    mask[flips] = ~mask[flips]
    if not mask.any():
        mask[rng.integers(len(mask))] = True
    hidden = genome["hidden_size"]
    if rng.random() < hidden_jump_prob:
        hidden = int(rng.choice(HIDDEN_SIZE_CHOICES))
    return {"mask": mask, "hidden_size": hidden}


def _tournament_select(rng, population, fitness, k=3):
    idx = rng.integers(0, len(population), size=k)
    return population[idx[np.argmin([fitness[i] for i in idx])]]


def run_ga(X_tr, cf_tr, cb_tr, y_tr, X_es, cf_es, cb_es, y_es, population_size=16, generations=12,
           elitism=2, seed=42, search_epochs=800, search_patience=100):
    rng = np.random.default_rng(seed)
    n_features = X_tr.shape[1]
    population = [_random_genome(rng, n_features) for _ in range(population_size)]
    history = []
    best_genome, best_loss = None, np.inf
    for gen in range(generations):
        scored = []
        for genome in population:
            loss, _ = _fit_genome(genome, X_tr, cf_tr, cb_tr, y_tr, X_es, cf_es, cb_es, y_es,
                                   search_epochs, search_patience, seed)
            scored.append(loss)
            if loss < best_loss:
                best_loss, best_genome = loss, genome
        history.append({"generation": gen, "best_es_mse": float(min(scored)), "mean_es_mse": float(np.mean(scored))})
        order = np.argsort(scored)
        new_population = [population[i] for i in order[:elitism]]
        while len(new_population) < population_size:
            p1 = _tournament_select(rng, population, scored)
            p2 = _tournament_select(rng, population, scored)
            new_population.append(_mutate(rng, _crossover(rng, p1, p2)))
        population = new_population
    return best_genome, best_loss, history


class GAEvolvedGate:
    def __init__(self, es_frac=0.2, seed=42, population_size=16, generations=12,
                 search_epochs=800, search_patience=100, final_epochs=3000, final_patience=200):
        self.es_frac, self.seed = es_frac, seed
        self.population_size, self.generations = population_size, generations
        self.search_epochs, self.search_patience = search_epochs, search_patience
        self.final_epochs, self.final_patience = final_epochs, final_patience
        self.net = None
        self.selected_mask_ = None
        self.selected_features_ = None
        self.ga_history_ = None
        self.train_time_sec = None

    def fit(self, val_df, item_popularity, cf_expert, cb_expert):
        start = time.perf_counter()
        feats = build_gate_features(val_df, item_popularity, cf_expert, cb_expert)
        X = feats.to_numpy(dtype=float)
        cf = val_df["cf_pred"].to_numpy(dtype=float)
        cb = val_df["cb_pred"].to_numpy(dtype=float)
        y = val_df["rating"].to_numpy(dtype=float)
        rng = np.random.default_rng(self.seed)
        idx = rng.permutation(len(val_df))
        n_es = int(round(len(val_df) * self.es_frac))
        es_idx, tr_idx = idx[:n_es], idx[n_es:]
        X_tr, X_es = X[tr_idx], X[es_idx]
        cf_tr, cf_es = cf[tr_idx], cf[es_idx]
        cb_tr, cb_es = cb[tr_idx], cb[es_idx]
        y_tr, y_es = y[tr_idx], y[es_idx]

        best_genome, best_loss, history = run_ga(
            X_tr, cf_tr, cb_tr, y_tr, X_es, cf_es, cb_es, y_es,
            population_size=self.population_size, generations=self.generations, seed=self.seed,
            search_epochs=self.search_epochs, search_patience=self.search_patience)
        self.ga_history_ = history
        self.selected_mask_ = best_genome["mask"]
        self.selected_features_ = [f for f, keep in zip(FEATURE_COLUMNS, self.selected_mask_) if keep]

        cols = np.where(self.selected_mask_)[0]
        l2 = 1.0 if best_genome["hidden_size"] == 0 else 1e-3
        self.net = GateNet(n_features=len(cols), hidden_size=best_genome["hidden_size"], l2=l2, lr=0.05, seed=self.seed)
        self.net.fit(X_tr[:, cols], cf_tr, cb_tr, y_tr, X_es[:, cols], cf_es, cb_es, y_es,
                      epochs=self.final_epochs, patience=self.final_patience)
        self.train_time_sec = time.perf_counter() - start
        return self

    def g(self, df, item_popularity, cf_expert, cb_expert) -> np.ndarray:
        feats = build_gate_features(df, item_popularity, cf_expert, cb_expert)
        X = feats.to_numpy(dtype=float)
        cols = np.where(self.selected_mask_)[0]
        return self.net.g(X[:, cols])

    def n_params(self) -> int:
        return self.net.n_params()

In [ ]:
print("Running GA search (feature subset + hidden_size) on VAL...")
ga_gate = GAEvolvedGate(population_size=16, generations=12, seed=RANDOM_SEED).fit(val_df, item_popularity, cf_expert, cb_expert)
print(f"  train_time_sec={ga_gate.train_time_sec:.2f}  n_params={ga_gate.n_params()}")
print(f"  selected features: {ga_gate.selected_features_}")
print(f"  hidden_size: {ga_gate.net.hidden_size}")

val_df["gate_g"] = ga_gate.g(val_df, item_popularity, cf_expert, cb_expert)
val_df["hybrid_pred"] = blend_scores(val_df["cf_pred"], val_df["cb_pred"], val_df["gate_g"])
test_df["gate_g"] = ga_gate.g(test_df, item_popularity, cf_expert, cb_expert)
test_df["hybrid_pred"] = blend_scores(test_df["cf_pred"], test_df["cb_pred"], test_df["gate_g"])

model6_val_metrics = segmented_rating_metrics(val_df, "rating", "hybrid_pred")
model6_test_metrics = segmented_rating_metrics(test_df, "rating", "hybrid_pred")
print(f"[test] RMSE(overall)={model6_test_metrics['overall']['rmse']:.4f}")

test_df.to_csv(PRED_DIR / "model6_ga_gate_test.csv", index=False)
pd.DataFrame(ga_gate.ga_history_).to_csv(METRICS_DIR / "model6_ga_fitness_curve.csv", index=False)
with open(METRICS_DIR / "model6_ga_gate.json", "w") as f:
    json.dump({"val": model6_val_metrics, "test": model6_test_metrics,
               "compute_cost": {"train_time_sec": ga_gate.train_time_sec, "n_params": ga_gate.n_params()},
               "selected_features": ga_gate.selected_features_, "hidden_size": ga_gate.net.hidden_size,
               "ga_history": ga_gate.ga_history_}, f, indent=2)

hist = pd.DataFrame(ga_gate.ga_history_)
plt.figure(figsize=(7, 4.5))
plt.plot(hist["generation"], hist["best_es_mse"], marker="o", label="best (elite) held-out MSE")
plt.plot(hist["generation"], hist["mean_es_mse"], marker="o", label="population mean held-out MSE")
plt.xlabel("GA generation"); plt.ylabel("Held-out hybrid MSE (lower is better)")
plt.title("Model 6: GA feature/architecture search convergence")
plt.legend(); plt.tight_layout()
plt.savefig(METRICS_DIR / "model6_ga_fitness_curve.png", dpi=150)
plt.show()

## Model 7 — Sequential Gate (BiLSTM over long-/short-term history)

A BiLSTM over each user's long-term and short-term TRAIN-only rating history, conditioned on the candidate item. A query needs `SEQ_MIN_HISTORY` prior TRAIN ratings strictly before its timestamp to run the LSTM at all; otherwise it falls back to Model 3's fixed alpha. Per history step the model sees: rating (normalized), an exponential recency decay, item log-popularity, and the item's content-similarity to the candidate item.

In [ ]:
def build_user_train_sequences(train_df: pd.DataFrame) -> dict:
    seqs = {}
    for uid, g in train_df.sort_values("timestamp").groupby("userId"):
        seqs[uid] = (g["timestamp"].to_numpy(), g["movieId"].to_numpy(), g["rating"].to_numpy(dtype=float))
    return seqs


def _prior_slice(seqs, user_id, before_ts, max_len):
    entry = seqs.get(user_id)
    if entry is None:
        return np.array([]), np.array([]), np.array([])
    timestamps, movie_ids, ratings = entry
    cut = bisect.bisect_left(timestamps, before_ts)
    lo = max(0, cut - max_len)
    return timestamps[lo:cut], movie_ids[lo:cut], ratings[lo:cut]


def prior_history_len(seqs, user_id, before_ts) -> int:
    entry = seqs.get(user_id)
    if entry is None:
        return 0
    return int(bisect.bisect_left(entry[0], before_ts))


def _step_features(cb_expert, item_popularity, query_ts, query_item_id, timestamps, movie_ids, ratings):
    n = len(ratings)
    if n == 0:
        return np.zeros((0, 4), dtype=np.float32)
    recency_days = (query_ts - timestamps) / 86400.0
    recency_decay = 0.5 ** (np.clip(recency_days, 0, None) / RECENCY_HALF_LIFE_DAYS)
    pop = np.array([item_popularity.get(m, 0) for m in movie_ids], dtype=float)
    log_pop = np.log1p(pop) / 10.0
    sims = np.zeros(n, dtype=float)
    if query_item_id in cb_expert.item_index_:
        target_vec = cb_expert.item_vectors[cb_expert.item_index_[query_item_id]]
        rows = [cb_expert.item_index_[m] for m in movie_ids if m in cb_expert.item_index_]
        keep = [i for i, m in enumerate(movie_ids) if m in cb_expert.item_index_]
        if rows:
            hist_vecs = cb_expert.item_vectors[rows]
            sims[keep] = np.asarray(hist_vecs.dot(target_vec.T).todense()).ravel()
    return np.stack([ratings / 5.0, recency_decay, log_pop, sims], axis=1).astype(np.float32)


def build_sequence_batch(df, seqs, cb_expert, item_popularity, user_col="userId", item_col="movieId", ts_col="timestamp"):
    n = len(df)
    long_len, short_len = SEQ_LONG_LEN, SEQ_SHORT_LEN
    long_feats = np.zeros((n, long_len, 4), dtype=np.float32)
    short_feats = np.zeros((n, short_len, 4), dtype=np.float32)
    long_lens = np.zeros(n, dtype=np.int64)
    short_lens = np.zeros(n, dtype=np.int64)
    fallback_mask = np.zeros(n, dtype=bool)
    for row_i, (uid, iid, ts) in enumerate(zip(df[user_col], df[item_col], df[ts_col])):
        prior_len = prior_history_len(seqs, uid, ts)
        fallback_mask[row_i] = prior_len < SEQ_MIN_HISTORY
        if fallback_mask[row_i]:
            continue
        timestamps, movie_ids, ratings = _prior_slice(seqs, uid, ts, long_len)
        feats = _step_features(cb_expert, item_popularity, ts, iid, timestamps, movie_ids, ratings)
        L = len(feats)
        long_feats[row_i, long_len - L:, :] = feats
        long_lens[row_i] = L
        s_timestamps, s_movie_ids, s_ratings = _prior_slice(seqs, uid, ts, short_len)
        s_feats = _step_features(cb_expert, item_popularity, ts, iid, s_timestamps, s_movie_ids, s_ratings)
        S = len(s_feats)
        short_feats[row_i, short_len - S:, :] = s_feats
        short_lens[row_i] = S
    return long_feats, long_lens, short_feats, short_lens, fallback_mask

In [ ]:
class BiLSTMGateNet(nn.Module):
    def __init__(self, n_gate_features, step_dim=4, hidden_size=16, mlp_hidden=16):
        super().__init__()
        self.long_lstm = nn.LSTM(step_dim, hidden_size, batch_first=True, bidirectional=True)
        self.short_lstm = nn.LSTM(step_dim, hidden_size, batch_first=True, bidirectional=True)
        rep_dim = 4 * hidden_size + n_gate_features
        self.head = nn.Sequential(nn.Linear(rep_dim, mlp_hidden), nn.Tanh(), nn.Linear(mlp_hidden, 1))

    @staticmethod
    def _encode(lstm, x, lengths):
        lengths_clamped = lengths.clamp(min=1)
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths_clamped.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = lstm(packed)
        rep = torch.cat([h_n[0], h_n[1]], dim=-1)
        rep = rep * (lengths > 0).float().unsqueeze(-1)
        return rep

    def forward(self, long_x, long_len, short_x, short_len, gate_feats):
        long_rep = self._encode(self.long_lstm, long_x, long_len)
        short_rep = self._encode(self.short_lstm, short_x, short_len)
        z = torch.cat([long_rep, short_rep, gate_feats], dim=-1)
        return torch.sigmoid(self.head(z)).squeeze(-1)


class SequentialGate:
    def __init__(self, fallback_alpha, seed=42, hidden_size=16, lr=1e-3, weight_decay=1e-4):
        self.fallback_alpha, self.seed, self.hidden_size = fallback_alpha, seed, hidden_size
        self.lr, self.weight_decay = lr, weight_decay
        self.net = None
        self.gate_feat_mean_ = None
        self.gate_feat_std_ = None
        self.train_time_sec = None
        self.history_ = []

    def fit(self, val_df, seqs, item_popularity, cf_expert, cb_expert, es_frac=0.2, epochs=60, patience=10, batch_size=512):
        start = time.perf_counter()
        torch.manual_seed(self.seed)

        gate_feats = build_gate_features(val_df, item_popularity, cf_expert, cb_expert).to_numpy(dtype=np.float32)
        long_feats, long_lens, short_feats, short_lens, fallback_mask = build_sequence_batch(val_df, seqs, cb_expert, item_popularity)

        keep = ~fallback_mask
        cf = val_df["cf_pred"].to_numpy(dtype=np.float32)[keep]
        cb = val_df["cb_pred"].to_numpy(dtype=np.float32)[keep]
        y = val_df["rating"].to_numpy(dtype=np.float32)[keep]
        gate_feats = gate_feats[keep]
        long_feats, long_lens = long_feats[keep], long_lens[keep]
        short_feats, short_lens = short_feats[keep], short_lens[keep]

        self.gate_feat_mean_ = gate_feats.mean(axis=0)
        self.gate_feat_std_ = gate_feats.std(axis=0)
        self.gate_feat_std_[self.gate_feat_std_ < 1e-8] = 1.0
        gate_feats_n = (gate_feats - self.gate_feat_mean_) / self.gate_feat_std_

        n = len(y)
        rng = np.random.default_rng(self.seed)
        idx = rng.permutation(n)
        n_es = int(round(n * es_frac))
        es_idx, tr_idx = idx[:n_es], idx[n_es:]

        def to_t(arr):
            return torch.as_tensor(arr)

        Xtr = dict(long_x=to_t(long_feats[tr_idx]), long_len=to_t(long_lens[tr_idx]),
                   short_x=to_t(short_feats[tr_idx]), short_len=to_t(short_lens[tr_idx]),
                   gate=to_t(gate_feats_n[tr_idx]).float(), cf=to_t(cf[tr_idx]), cb=to_t(cb[tr_idx]), y=to_t(y[tr_idx]))
        Xes = dict(long_x=to_t(long_feats[es_idx]), long_len=to_t(long_lens[es_idx]),
                   short_x=to_t(short_feats[es_idx]), short_len=to_t(short_lens[es_idx]),
                   gate=to_t(gate_feats_n[es_idx]).float(), cf=to_t(cf[es_idx]), cb=to_t(cb[es_idx]), y=to_t(y[es_idx]))

        self.net = BiLSTMGateNet(n_gate_features=len(FEATURE_COLUMNS), hidden_size=self.hidden_size)
        opt = torch.optim.Adam(self.net.parameters(), lr=self.lr, weight_decay=self.weight_decay)

        n_tr = len(tr_idx)
        best_es, best_state, no_improve = np.inf, None, 0
        for epoch in range(1, epochs + 1):
            self.net.train()
            perm = torch.randperm(n_tr)
            for start_i in range(0, n_tr, batch_size):
                b = perm[start_i:start_i + batch_size]
                opt.zero_grad()
                g = self.net(Xtr["long_x"][b], Xtr["long_len"][b], Xtr["short_x"][b], Xtr["short_len"][b], Xtr["gate"][b])
                pred = g * Xtr["cf"][b] + (1 - g) * Xtr["cb"][b]
                loss = torch.mean((pred - Xtr["y"][b]) ** 2)
                loss.backward()
                opt.step()

            self.net.eval()
            with torch.no_grad():
                g_tr = self.net(Xtr["long_x"], Xtr["long_len"], Xtr["short_x"], Xtr["short_len"], Xtr["gate"])
                train_mse = torch.mean((g_tr * Xtr["cf"] + (1 - g_tr) * Xtr["cb"] - Xtr["y"]) ** 2).item()
                g_es = self.net(Xes["long_x"], Xes["long_len"], Xes["short_x"], Xes["short_len"], Xes["gate"])
                es_mse = torch.mean((g_es * Xes["cf"] + (1 - g_es) * Xes["cb"] - Xes["y"]) ** 2).item()
            self.history_.append((epoch, train_mse, es_mse))

            if es_mse < best_es - 1e-5:
                best_es, best_state, no_improve = es_mse, {k: v.clone() for k, v in self.net.state_dict().items()}, 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    break

        if best_state is not None:
            self.net.load_state_dict(best_state)
        self.net.eval()
        self.train_time_sec = time.perf_counter() - start
        return self

    def g(self, df, seqs, item_popularity, cf_expert, cb_expert):
        gate_feats = build_gate_features(df, item_popularity, cf_expert, cb_expert).to_numpy(dtype=np.float32)
        long_feats, long_lens, short_feats, short_lens, fallback_mask = build_sequence_batch(df, seqs, cb_expert, item_popularity)
        g_out = np.full(len(df), self.fallback_alpha, dtype=np.float32)
        keep = ~fallback_mask
        if keep.any():
            gate_feats_n = (gate_feats[keep] - self.gate_feat_mean_) / self.gate_feat_std_
            with torch.no_grad():
                g_lstm = self.net(
                    torch.as_tensor(long_feats[keep]), torch.as_tensor(long_lens[keep]),
                    torch.as_tensor(short_feats[keep]), torch.as_tensor(short_lens[keep]),
                    torch.as_tensor(gate_feats_n).float()).numpy()
            g_out[keep] = g_lstm
        return g_out, fallback_mask

    def n_params(self) -> int:
        return int(sum(p.numel() for p in self.net.parameters()))

In [ ]:
seqs = build_user_train_sequences(train_df)
print(f"Fallback alpha (Model 3): {static_gate.alpha:.2f}")
print("Training Sequential Gate (BiLSTM) on VAL...")
seq_gate = SequentialGate(fallback_alpha=static_gate.alpha, seed=RANDOM_SEED).fit(
    val_df, seqs, item_popularity, cf_expert, cb_expert, epochs=60, patience=10)
print(f"  train_time_sec={seq_gate.train_time_sec:.2f}  n_params={seq_gate.n_params()}")

val_g, val_fallback = seq_gate.g(val_df, seqs, item_popularity, cf_expert, cb_expert)
val_df["gate_g"], val_df["used_fallback"] = val_g, val_fallback
val_df["hybrid_pred"] = blend_scores(val_df["cf_pred"], val_df["cb_pred"], val_g)

test_g, test_fallback = seq_gate.g(test_df, seqs, item_popularity, cf_expert, cb_expert)
test_df["gate_g"], test_df["used_fallback"] = test_g, test_fallback
test_df["hybrid_pred"] = blend_scores(test_df["cf_pred"], test_df["cb_pred"], test_g)

model7_val_metrics = segmented_rating_metrics(val_df, "rating", "hybrid_pred")
model7_test_metrics = segmented_rating_metrics(test_df, "rating", "hybrid_pred")
print(f"[test] RMSE(overall)={model7_test_metrics['overall']['rmse']:.4f}  "
      f"fallback rate: {test_fallback.mean():.1%}")

fallback_metrics = {}
for used_fallback, sub in test_df.groupby("used_fallback"):
    key = "fallback_to_static" if used_fallback else "genuine_bilstm"
    fallback_metrics[key] = segmented_rating_metrics(sub, "rating", "hybrid_pred")

test_df.to_csv(PRED_DIR / "model7_sequential_gate_test.csv", index=False)
with open(METRICS_DIR / "model7_sequential_gate.json", "w") as f:
    json.dump({"val": model7_val_metrics, "test": model7_test_metrics, "test_by_fallback": fallback_metrics,
               "test_fallback_rate": float(test_fallback.mean()),
               "compute_cost": {"train_time_sec": seq_gate.train_time_sec, "n_params": seq_gate.n_params()},
               "training_curve_tail": seq_gate.history_[-5:], "seq_min_history": SEQ_MIN_HISTORY,
               "fallback_alpha": static_gate.alpha}, f, indent=2)

## Full 7-model comparison

Consolidates every model's TEST predictions onto one row-aligned table: segmented RMSE/MAE + ranking metrics (Spearman/Kendall/NDCG/ARHR/ROC-AUC), a compute-cost table, and pairwise statistical significance (paired bootstrap + Wilcoxon) between every model pair, overall and per sparsity segment.

In [ ]:
experts_test_df = pd.read_csv(PRED_DIR / "experts_test.csv")
MODEL_PRED_SOURCES = {
    "1_CF_SVDpp": (experts_test_df, "cf_pred"),
    "2_ContentBased": (experts_test_df, "cb_pred"),
    "3_StaticHybrid": (pd.read_csv(PRED_DIR / "model3_static_hybrid_test.csv"), "hybrid_pred"),
    "4_LearnedGate": (pd.read_csv(PRED_DIR / "model4_learned_gate_test.csv"), "hybrid_pred"),
    "5_BanditGate": (pd.read_csv(PRED_DIR / "model5_bandit_gate_test.csv"), "hybrid_pred"),
    "6_GAEvolvedGate": (pd.read_csv(PRED_DIR / "model6_ga_gate_test.csv"), "hybrid_pred"),
    "7_SequentialGate": (pd.read_csv(PRED_DIR / "model7_sequential_gate_test.csv"), "hybrid_pred"),
}

master = experts_test_df[["userId", "movieId", "rating", "segment", "train_rating_count"]].copy()
accuracy, ranking = {}, {}
for name, (df, col) in MODEL_PRED_SOURCES.items():
    assert (df["userId"].to_numpy() == experts_test_df["userId"].to_numpy()).all(), f"{name} not row-aligned"
    assert (df["movieId"].to_numpy() == experts_test_df["movieId"].to_numpy()).all(), f"{name} not row-aligned"
    master[f"pred__{name}"] = df[col].to_numpy()
    scored = experts_test_df.copy()
    scored["pred"] = df[col].to_numpy()
    accuracy[name] = segmented_rating_metrics(scored, "rating", "pred")
    ranking[name] = segmented_ranking_metrics(scored, "rating", "pred")
    print(f"{name:20s} RMSE(overall)={accuracy[name]['overall']['rmse']:.4f}  "
          f"NDCG(overall)={ranking[name]['overall']['ndcg']:.4f}  "
          f"Spearman(overall)={ranking[name]['overall']['spearman']:.4f}")

In [ ]:
COMPUTE_COST_SOURCES = {
    "1_CF_SVDpp": experts_results["compute_cost"]["cf_svdpp"],
    "2_ContentBased": experts_results["compute_cost"]["content_based"],
    "3_StaticHybrid": {"train_time_sec": static_gate.train_time_sec, "n_params": static_gate.n_params()},
    "4_LearnedGate": {"train_time_sec": learned_gate.train_time_sec, "n_params": learned_gate.n_params()},
    "5_BanditGate": {"train_time_sec": bandit_train_time_sec, "n_params": bandit.n_params()},
    "6_GAEvolvedGate": {"train_time_sec": ga_gate.train_time_sec, "n_params": ga_gate.n_params()},
    "7_SequentialGate": {"train_time_sec": seq_gate.train_time_sec, "n_params": seq_gate.n_params()},
}
print("\nCompute cost:")
for name, cc in COMPUTE_COST_SOURCES.items():
    print(f"  {name:20s} train_time_sec={cc['train_time_sec']:.3f}  n_params={cc['n_params']}")

print("\nRunning pairwise significance tests (paired bootstrap + Wilcoxon)...")
pred_cols = {name: f"pred__{name}" for name in MODEL_PRED_SOURCES}
sig_df = compare_models(master, true_col="rating", model_preds=pred_cols, segment_col="segment", n_boot=2000)
sig_df.to_csv(METRICS_DIR / "full_comparison_significance.csv", index=False)

key_pairs = sig_df[(sig_df["segment"] == "overall") &
                    (((sig_df["model_a"] == "3_StaticHybrid") & (sig_df["model_b"].str.startswith(("4_", "5_", "6_", "7_")))) |
                     ((sig_df["model_b"] == "3_StaticHybrid") & (sig_df["model_a"].str.startswith(("4_", "5_", "6_", "7_")))))]
print("\nModel 3 (Static Hybrid) vs each adaptive gate, overall test set:")
print(key_pairs[["model_a", "model_b", "rmse_a", "rmse_b", "mean_diff", "bootstrap_significant", "wilcoxon_p"]].to_string(index=False))

accuracy_table = []
for name in MODEL_PRED_SOURCES:
    for seg in ("overall", "cold", "warm", "power"):
        if seg in accuracy[name]:
            row = {"model": name, "segment": seg}
            row.update(accuracy[name][seg])
            row.update({f"rank_{k}": v for k, v in ranking[name][seg].items()})
            accuracy_table.append(row)
accuracy_df = pd.DataFrame(accuracy_table)
accuracy_df.to_csv(METRICS_DIR / "full_comparison_table.csv", index=False)
master.to_csv(PRED_DIR / "full_comparison_predictions.csv", index=False)

with open(METRICS_DIR / "full_comparison.json", "w") as f:
    json.dump({"accuracy": accuracy, "ranking": ranking, "compute_cost": COMPUTE_COST_SOURCES}, f, indent=2)

accuracy_df[accuracy_df["segment"] == "overall"][["model", "rmse", "mae", "n"]].set_index("model")

## Multi-seed robustness check (5 seeds)

A single seed can't distinguish a real effect from noise in the stochastic simulated cold-start split and in gate training/GA search/bandit exploration. This re-runs the entire pipeline — split → experts → every gate — under 5 seeds and reports mean ± std segmented RMSE per model.

In [ ]:
SEEDS = [42, 1, 2, 3, 4]


def run_pipeline_for_seed(seed: int) -> dict:
    tr_df, v_df, te_df = build_splits(seed=seed)
    u_segments = build_user_segments(tr_df)
    i_pop = build_item_popularity(tr_df)

    cf = CFExpertSVDpp(random_state=seed).fit(tr_df)
    cb = ContentBasedExpert().fit(tr_df, movies_df, tags_df)

    def score(df):
        out = attach_segments(df.copy(), u_segments)
        out["cf_pred"] = cf.predict_batch(out)
        out["cb_pred"] = cb.predict_batch(out)
        return out

    v_s, te_s = score(v_df), score(te_df)
    results = {}

    def record(name, pred_col):
        results[name] = segmented_rating_metrics(te_s, "rating", pred_col)

    te_s["m1"] = te_s["cf_pred"]; record("1_CF_SVDpp", "m1")
    te_s["m2"] = te_s["cb_pred"]; record("2_ContentBased", "m2")

    static = StaticGate().fit(v_s, step=0.02)
    te_s["m3"] = blend_scores(te_s["cf_pred"], te_s["cb_pred"], static.g(len(te_s)))
    record("3_StaticHybrid", "m3")

    learned = LearnedGate(hidden_size=0, l2=1.0, es_frac=0.2, seed=seed).fit(v_s, i_pop, cf, cb)
    te_s["m4"] = blend_scores(te_s["cf_pred"], te_s["cb_pred"], learned.g(te_s, i_pop, cf, cb))
    record("4_LearnedGate", "m4")

    v_sorted = v_s.sort_values("timestamp").reset_index(drop=True)
    Xv = build_gate_features(v_sorted, i_pop, cf, cb).to_numpy(dtype=float)
    b = ContextualBanditGate(n_features=len(FEATURE_COLUMNS), strategy="ucb", seed=seed)
    run_stream(b, Xv, v_sorted["cf_pred"].to_numpy(dtype=float), v_sorted["cb_pred"].to_numpy(dtype=float),
               v_sorted["rating"].to_numpy(dtype=float), explore=True)
    Xt = build_gate_features(te_s, i_pop, cf, cb).to_numpy(dtype=float)
    preds5, _, _ = run_stream(b, Xt, te_s["cf_pred"].to_numpy(dtype=float), te_s["cb_pred"].to_numpy(dtype=float),
                               te_s["rating"].to_numpy(dtype=float), explore=False)
    te_s["m5"] = preds5
    record("5_BanditGate", "m5")

    ga = GAEvolvedGate(population_size=16, generations=12, seed=seed).fit(v_s, i_pop, cf, cb)
    te_s["m6"] = blend_scores(te_s["cf_pred"], te_s["cb_pred"], ga.g(te_s, i_pop, cf, cb))
    record("6_GAEvolvedGate", "m6")

    s = build_user_train_sequences(tr_df)
    seq_g = SequentialGate(fallback_alpha=static.alpha, seed=seed).fit(v_s, s, i_pop, cf, cb, epochs=60, patience=10)
    g7, _ = seq_g.g(te_s, s, i_pop, cf, cb)
    te_s["m7"] = blend_scores(te_s["cf_pred"], te_s["cb_pred"], g7)
    record("7_SequentialGate", "m7")

    return results


def aggregate_metrics(metrics_list):
    rmses = [m["rmse"] for m in metrics_list]
    maes = [m["mae"] for m in metrics_list]
    ns = [m["n"] for m in metrics_list]
    return {"rmse_mean": float(np.mean(rmses)), "rmse_std": float(np.std(rmses)),
            "mae_mean": float(np.mean(maes)), "mae_std": float(np.std(maes)),
            "n_mean": float(np.mean(ns)), "n_seeds": len(metrics_list)}


def aggregate_segmented_metrics(list_of_segmented):
    segments = set()
    for d in list_of_segmented:
        segments.update(d.keys())
    return {seg: aggregate_metrics([d[seg] for d in list_of_segmented if seg in d]) for seg in segments}

In [ ]:
print(f"Running full 7-model pipeline across {len(SEEDS)} seeds: {SEEDS}")
per_seed = {}
for seed in SEEDS:
    start = time.perf_counter()
    results = run_pipeline_for_seed(seed)
    elapsed = time.perf_counter() - start
    for name, m in results.items():
        per_seed.setdefault(name, []).append(m)
    print(f"  seed={seed} done in {elapsed:.1f}s  overall_rmse: " +
          ", ".join(f"{n.split('_', 1)[1]}={m['overall']['rmse']:.3f}" for n, m in results.items()))

multiseed_aggregate = {name: aggregate_segmented_metrics(metrics_list) for name, metrics_list in per_seed.items()}

print("\n[Multi-seed summary, test set, mean +/- std over 5 seeds]")
print(f"  {'model':20s} {'overall':>16s} {'cold':>16s} {'warm':>16s} {'power':>16s}")
for name, agg in multiseed_aggregate.items():
    cells = []
    for seg in ("overall", "cold", "warm", "power"):
        cells.append(f"{agg[seg]['rmse_mean']:.3f}+/-{agg[seg]['rmse_std']:.3f}" if seg in agg else "n/a")
    print(f"  {name:20s} " + " ".join(f"{c:>16s}" for c in cells))

with open(METRICS_DIR / "multiseed_full_comparison.json", "w") as f:
    json.dump({"seeds": SEEDS, "per_seed": per_seed, "aggregate": multiseed_aggregate}, f, indent=2)
print(f"\nSaved -> {METRICS_DIR / 'multiseed_full_comparison.json'}")